# October Milestone — Task 1: Encoder, Tokenizer, and Model Pre-registration

### I'll take these steps for this Task:

Review dependencies and lock inputs

Choose and justify the encoder/tokenizer

Run token-length and truncation audit

Pre-register training configuration

Define model selection and threshold rules

Document and freeze the setup

## Step 1: Load Required Project Data

This notebook pre-registers the encoder, tokenizer, and training configuration that will be used for the next modeling stage.

To keep the analysis consistent with the September pipeline, we load:

- The frozen training split
- The final 10 CUAD categories
- The approved paragraph-based training chunks

These are the same core inputs used by the September classification pipeline.
The loaded data will be used to:
1. Compare candidate encoders and tokenizers.
2. Audit token lengths and possible truncation.
3. Determine whether candidate models fit our compute and input-length requirements.

In [49]:
# Load shared project helper files into the Colab runtime

# These files already contain helper functions used throughout
# the team's September pipeline.
#
# Downloading them from main ensures this notebook uses the
# current shared project code instead of creating duplicate
# helper functions.

!wget -q \
https://raw.githubusercontent.com/Break-Through-Tech/Accenture-1O-contract-review-challenge/main/notebooks/utils.py \
-O utils.py

!wget -q \
https://raw.githubusercontent.com/Break-Through-Tech/Accenture-1O-contract-review-challenge/main/notebooks/chunking_helper.py \
-O chunking_helper.py


# Confirm that Colab can see both files.
import os

print("utils.py available:", os.path.exists("utils.py"))
print("chunking_helper.py available:", os.path.exists("chunking_helper.py"))

utils.py available: True
chunking_helper.py available: True


In [50]:
# Imports

import pandas as pd
import numpy as np
from pathlib import Path

# Shared project utilities
from utils import load_file_from_github, load_split_data


# Final 10 CUAD categories
#
# Keep the category order consistent with Tasks 12 and 13.
# These are the current frozen categories from the September
# milestone.
#
# NOTE:
# The team may revisit the category-selection analysis later as Beth advised,
# but this Task 1 analysis should initially remain consistent
# with the currently approved modeling pipeline.

FINAL_IDS = [
    "governing_law",
    "renewal_term",
    "revenue_profit_sharing",
    "cap_on_liability",
    "uncapped_liability",
    "termination_for_convenience",
    "anti_assignment",
    "audit_rights",
    "license_grant",
    "exclusivity",
]

print("Number of selected categories:", len(FINAL_IDS))

Number of selected categories: 10


In [51]:
# Load frozen training data directly from GitHub

BASE_URL = (
    "https://raw.githubusercontent.com/"
    "Break-Through-Tech/Accenture-1O-contract-review-challenge/"
    "main/"
)

# Load normalized project tables
documents = pd.read_parquet(
    BASE_URL + "notebooks/split_data/documents.parquet"
)

annotation_sets = pd.read_parquet(
    BASE_URL + "notebooks/split_data/annotation_sets.parquet"
)

spans = pd.read_parquet(
    BASE_URL + "notebooks/split_data/spans.parquet"
)

categories = pd.read_parquet(
    BASE_URL + "notebooks/split_data/categories.parquet"
)

# Load the frozen train/validation split
split_map = pd.read_parquet(
    BASE_URL + "notebooks/frozen-split_data/frozen-split.parquet"
)


# Apply the frozen split

documents_with_split = documents.merge(
    split_map,
    on="context_group_id",
    how="left"
)

# Keep only training documents
train_documents_all = documents_with_split[
    documents_with_split["split"] == "train"
].copy()

# Get training contract IDs
train_contract_ids = set(
    train_documents_all["contract_id"]
)

# Keep only annotation sets from training contracts
train_annotation_sets_all = annotation_sets[
    annotation_sets["contract_id"].isin(train_contract_ids)
].copy()

# Keep only spans from training annotation sets
train_spans_all = spans[
    spans["annotation_set_id"].isin(
        train_annotation_sets_all["annotation_set_id"]
    )
].copy()

train_categories = categories.copy()


# Verification

print("Training documents:", train_documents_all.shape)
print("Training annotation sets:", train_annotation_sets_all.shape)
print("Training spans:", train_spans_all.shape)
print("Categories:", train_categories.shape)

Training documents: (325, 5)
Training annotation sets: (13325, 4)
Training spans: (8964, 6)
Categories: (41, 3)


In [52]:
# Keep only data associated with the final 10 categories

# Find annotation sets belonging to the selected categories.
train_sets = train_annotation_sets_all[
    train_annotation_sets_all["category_id"].isin(FINAL_IDS)
].copy()


# Keep documents connected to those annotation sets.
train_documents = train_documents_all[
    train_documents_all["contract_id"].isin(
        train_sets["contract_id"]
    )
].copy()


# Attach category and contract information to the annotated spans.
train_spans = train_spans_all.merge(
    train_sets[
        [
            "annotation_set_id",
            "category_id",
            "contract_id"
        ]
    ],
    on="annotation_set_id",
    how="inner"
)


# Verification

print("Final categories:", train_sets["category_id"].nunique())
print("Training documents:", len(train_documents))
print("Selected annotation sets:", len(train_sets))
print("Selected annotated spans:", len(train_spans))

print("\nCategories represented:")
print(
    sorted(train_sets["category_id"].unique())
)

Final categories: 10
Training documents: 325
Selected annotation sets: 3250
Selected annotated spans: 3078

Categories represented:
['anti_assignment', 'audit_rights', 'cap_on_liability', 'exclusivity', 'governing_law', 'license_grant', 'renewal_term', 'revenue_profit_sharing', 'termination_for_convenience', 'uncapped_liability']


In [53]:
# Load the approved paragraph-based training chunks

CHUNKS_URL = (
    "https://raw.githubusercontent.com/"
    "Break-Through-Tech/"
    "Accenture-1O-contract-review-challenge/"
    "main/notebooks/task10_outputs/chunks.parquet"
)

train_chunks = pd.read_parquet(CHUNKS_URL)


# Basic verification

print("Training chunks:", train_chunks.shape)

print("\nChunk columns:")
print(train_chunks.columns.tolist())

display(train_chunks.head())

Training chunks: (10841, 11)

Chunk columns:
['method', 'contract_id', 'context_group_id', 'chunk_id', 'chunk_number', 'char_start', 'char_end', 'num_tokens', 'chunk_text', 'chunk_hash', 'document_hash']


,method,contract_id,context_group_id,chunk_id,chunk_number,char_start,char_end,num_tokens,chunk_text,chunk_hash,document_hash
0,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0000,0,0,2056,463,EXHIBIT 10.6\n\n ...,8d9734236056eaa8a3619bf2942ca9b27adadb920c52ff...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...
1,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0001,1,2076,3126,512,1.1 Grant and Acceptance. Company h...,0eb8801fa97ea183343a724bb6fbd0caab779ccfcdb942...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...
2,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0002,2,2979,3717,359,such customers in the Market to those custom...,4e070f16a388ecff6151b6344f9ff838669ee88d9e2424...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...
3,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0003,3,3737,4788,512,1.2 License. The Company hereby grants ...,fe8b1735b9cc0a5536b6771e0f47e30c7c80c93992179d...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...
4,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0004,4,4658,5228,277,sublicense the Names or to do ...,c2b10a8c277aef96161b58f42a2068a32e017068e12f31...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...


### Review Dependencies and Lock Inputs

Before selecting a new encoder or tokenizer, this step confirms the fixed inputs inherited from the September milestone.

The October model work will initially use:

- **Final 10 CUAD categories** selected during Task 5
- **Paragraph-based chunking** selected during Task 9
- **Frozen train/validation split** established earlier in the pipeline
- **Task 12 logistic-regression classifier** as the current baseline

The purpose of this step is to prevent later modeling decisions from unintentionally changing the label set, chunking strategy, or evaluation split.

The official test set from Task 13 is not used for model selection. Its results are retained only as a reference for the current baseline's generalization performance.

In [54]:
# 1. Lock the final 10 CUAD categories

FINAL_IDS = [
    "governing_law",
    "renewal_term",
    "revenue_profit_sharing",
    "cap_on_liability",
    "uncapped_liability",
    "termination_for_convenience",
    "anti_assignment",
    "audit_rights",
    "license_grant",
    "exclusivity",
]

# Verify that exactly 10 unique categories are registered.
assert len(FINAL_IDS) == 10
assert len(set(FINAL_IDS)) == 10

# Confirm that all selected categories exist in the project category table.
available_categories = set(categories["category_id"])

missing_categories = [
    category
    for category in FINAL_IDS
    if category not in available_categories
]

assert not missing_categories, (
    f"Selected categories missing from category table: {missing_categories}"
)

print("PASS: Final 10 CUAD categories confirmed.")

for i, category in enumerate(FINAL_IDS, start=1):
    print(f"{i}. {category}")

PASS: Final 10 CUAD categories confirmed.
1. governing_law
2. renewal_term
3. revenue_profit_sharing
4. cap_on_liability
5. uncapped_liability
6. termination_for_convenience
7. anti_assignment
8. audit_rights
9. license_grant
10. exclusivity


In [55]:
# 2. Confirm the frozen train/validation split
split_counts = (
    split_map["split"]
    .value_counts()
    .rename_axis("split")
    .reset_index(name="context_groups")
)

display(split_counts)


# Confirm that only the expected split labels exist.
assert set(split_map["split"].unique()) == {
    "train",
    "validation"
}

# Confirm that every context group appears only once.
assert split_map["context_group_id"].is_unique

# Count train and validation context groups.
n_train_groups = (
    split_map.loc[
        split_map["split"] == "train",
        "context_group_id"
    ].nunique()
)

n_validation_groups = (
    split_map.loc[
        split_map["split"] == "validation",
        "context_group_id"
    ].nunique()
)

print("Training context groups:", n_train_groups)
print("Validation context groups:", n_validation_groups)
print("Total context groups:", len(split_map))

print("\nPASS: Frozen split confirmed.")

,split,context_groups
0,train,324
1,validation,83


Training context groups: 324
Validation context groups: 83
Total context groups: 407

PASS: Frozen split confirmed.


In [56]:
# 3. Confirm the approved paragraph-based chunking setup

print("Chunk rows:", len(train_chunks))
print("Chunk columns:", train_chunks.columns.tolist())


# Check whether the method column is available.
if "method" in train_chunks.columns:
    print("\nChunking methods present:")
    print(train_chunks["method"].value_counts())


# Verify that every chunk belongs to one contract.
assert train_chunks["contract_id"].notna().all()


# Verify that character boundaries are valid.
assert (
    train_chunks["char_end"]
    > train_chunks["char_start"]
).all()


# Verify that the recorded token count does not exceed
# the 512-token model limit used by the September pipeline.
if "num_tokens" in train_chunks.columns:
    max_chunk_tokens = train_chunks["num_tokens"].max()

    print("\nMaximum recorded chunk length:", max_chunk_tokens)

    assert max_chunk_tokens <= 512, (
        "At least one approved chunk exceeds 512 tokens."
    )


print("\nPASS: Approved paragraph chunks confirmed.")

Chunk rows: 10841
Chunk columns: ['method', 'contract_id', 'context_group_id', 'chunk_id', 'chunk_number', 'char_start', 'char_end', 'num_tokens', 'chunk_text', 'chunk_hash', 'document_hash']

Chunking methods present:
method
paragraph_based    10841
Name: count, dtype: int64

Maximum recorded chunk length: 512

PASS: Approved paragraph chunks confirmed.


In [57]:
# 4. Register the current Task 12 classifier baseline

baseline_config = {
    "encoder": "sentence-transformers/all-distilroberta-v1",
    "classifier": "LogisticRegressionByCategory",
    "C": 10,
    "class_weight": None,
    "threshold": 0.20,
    "random_state": 42,

    # Task 12 validation results
    "validation_macro_precision": 0.505,
    "validation_macro_recall": 0.582,
    "validation_macro_f1": 0.535,
    "validation_pr_auc": 0.529,

    # Task 13 official-test reference results
    "official_test_macro_precision": 0.4636,
    "official_test_macro_recall": 0.5813,
    "official_test_macro_f1": 0.5112,
}

baseline_df = pd.DataFrame(
    baseline_config.items(),
    columns=["setting", "value"]
)

display(baseline_df)

,setting,value
0,encoder,sentence-transformers/all-distilroberta-v1
1,classifier,LogisticRegressionByCategory
2,C,10
3,class_weight,None
4,threshold,0.2
5,random_state,42
6,validation_macro_precision,0.505
7,validation_macro_recall,0.582
8,validation_macro_f1,0.535
9,validation_pr_auc,0.529


### Step 1 Conclusion

The modeling inputs are now locked for pre-registration:

- **Labels:** 10 selected CUAD categories
- **Chunking:** Paragraph-based chunks from the approved September pipeline
- **Split:** Frozen train/validation split at the context-group level
- **Current baseline:** DistilRoBERTa sentence embeddings + per-category logistic regression
- **Baseline validation Macro-F1:** 0.535
- **Official-test Macro-F1:** 0.5112, retained for reference only

These inputs will remain fixed while candidate encoder/tokenizer configurations are evaluated in the next step.

> Note: The team may later revisit Task 5 using the full original dataset, but this notebook initially preserves the currently approved 10-category setup so that encoder comparisons are made against a stable target.

## Step 2: Choose and Justify the Encoder / Tokenizer

This step compares a small set of realistic encoder/tokenizer options for the next modeling stage.

The candidates are:

1. **DistilRoBERTa**
2. **RoBERTa-base**
3. **Legal-BERT**

They are compared using three criteria:

- **Legal-text fit:** How well the pretrained model matches contract/legal language.
- **Compute fit:** Whether the model is realistic to train in IDEs: local IDEs and Google Colab environment.
- **Ease of training:** Compatibility with Hugging Face, 512-token inputs, and later parameter-efficient fine-tuning such as LoRA.

The goal is not to train all three models in this task. Instead, we pre-register the most practical encoder/tokenizer before downstream training begins.

In [58]:
# Register candidate encoders/tokenizers

# We compare only a small number of realistic models.
# This keeps the pre-registration focused and avoids
# turning Task 1 into a large hyperparameter/model search.

MODEL_CANDIDATES = {
    "DistilRoBERTa": {
        "model_id": "distilbert/distilroberta-base",
        "domain": "General English",
        "parameters_millions": 82,
        "layers": 6,
        "expected_max_length": 512
    },

    "RoBERTa-base": {
        "model_id": "FacebookAI/roberta-base",
        "domain": "General English",
        "parameters_millions": 125,
        "layers": 12,
        "expected_max_length": 512
    },

    "Legal-BERT": {
        "model_id": "nlpaueb/legal-bert-base-uncased",
        "domain": "Legal English",
        "parameters_millions": 110,
        "layers": 12,
        "expected_max_length": 512
    }
}

candidate_df = pd.DataFrame(
    MODEL_CANDIDATES
).T.reset_index(names="encoder")

display(candidate_df)

,encoder,model_id,domain,parameters_millions,layers,expected_max_length
0,DistilRoBERTa,distilbert/distilroberta-base,General English,82,6,512
1,RoBERTa-base,FacebookAI/roberta-base,General English,125,12,512
2,Legal-BERT,nlpaueb/legal-bert-base-uncased,Legal English,110,12,512


In [59]:
# Load candidate tokenizers

from transformers import AutoTokenizer

candidate_tokenizers = {}

for encoder_name, config in MODEL_CANDIDATES.items():

    print(f"Loading tokenizer: {encoder_name}")

    tokenizer = AutoTokenizer.from_pretrained(
        config["model_id"],
        use_fast=True
    )

    candidate_tokenizers[encoder_name] = tokenizer


print("\nPASS: All candidate tokenizers loaded.")

Loading tokenizer: DistilRoBERTa
Loading tokenizer: RoBERTa-base
Loading tokenizer: Legal-BERT

PASS: All candidate tokenizers loaded.


In [60]:
# Compare tokenizer properties

tokenizer_records = []

for encoder_name, tokenizer in candidate_tokenizers.items():

    tokenizer_records.append({
        "encoder": encoder_name,
        "model_id": MODEL_CANDIDATES[encoder_name]["model_id"],
        "vocab_size": tokenizer.vocab_size,
        "model_max_length": tokenizer.model_max_length,
        "special_tokens": tokenizer.num_special_tokens_to_add(
            pair=False
        )
    })


tokenizer_comparison = pd.DataFrame(
    tokenizer_records
)

display(tokenizer_comparison)

,encoder,model_id,vocab_size,model_max_length,special_tokens
0,DistilRoBERTa,distilbert/distilroberta-base,50265,512,2
1,RoBERTa-base,FacebookAI/roberta-base,50265,512,2
2,Legal-BERT,nlpaueb/legal-bert-base-uncased,30522,512,2


In [61]:
# Qualitative model comparison

encoder_comparison = pd.DataFrame([
    {
        "encoder": "DistilRoBERTa",
        "legal_text_fit": "Moderate",
        "compute_fit": "Strong",
        "ease_of_training": "Strong",
        "strength": "Smallest and fastest option; consistent with prior project work.",
        "tradeoff": "Not pretrained specifically on legal text."
    },

    {
        "encoder": "RoBERTa-base",
        "legal_text_fit": "Moderate",
        "compute_fit": "Moderate",
        "ease_of_training": "Strong",
        "strength": "Larger general-language encoder with more capacity.",
        "tradeoff": "More compute than DistilRoBERTa without legal-domain pretraining."
    },

    {
        "encoder": "Legal-BERT",
        "legal_text_fit": "Strong",
        "compute_fit": "Moderate",
        "ease_of_training": "Strong",
        "strength": "Pretrained on legal text, including contracts.",
        "tradeoff": "Larger than DistilRoBERTa and therefore more expensive to fine-tune."
    }
])

display(encoder_comparison)

,encoder,legal_text_fit,compute_fit,ease_of_training,strength,tradeoff
0,DistilRoBERTa,Moderate,Strong,Strong,Smallest and fastest option; consistent with p...,Not pretrained specifically on legal text.
1,RoBERTa-base,Moderate,Moderate,Strong,Larger general-language encoder with more capa...,More compute than DistilRoBERTa without legal-...
2,Legal-BERT,Strong,Moderate,Strong,"Pretrained on legal text, including contracts.",Larger than DistilRoBERTa and therefore more e...


In [62]:
# Preliminary ranking before token-length audit

preliminary_ranking = pd.DataFrame([
    {
        "rank": 1,
        "encoder": "Legal-BERT",
        "reason": (
            "Best match for legal contract language while remaining a standard 512-token Hugging Face model."
        )
    },
    {
        "rank": 2,
        "encoder": "DistilRoBERTa",
        "reason": (
            "Best compute fit and fastest option; also provides continuity with the September pipeline."
        )
    },
    {
        "rank": 3,
        "encoder": "RoBERTa-base",
        "reason": (
            "Strong general-purpose model, but requires more compute than DistilRoBERTa without Legal-BERT's domain advantage."
        )
    }
])

display(preliminary_ranking)

,rank,encoder,reason
0,1,Legal-BERT,Best match for legal contract language while r...
1,2,DistilRoBERTa,Best compute fit and fastest option; also prov...
2,3,RoBERTa-base,"Strong general-purpose model, but requires mor..."


### Step 2 Conclusion

**Legal-BERT** is the leading candidate because its pretraining is directly aligned with legal and contract language.

**DistilRoBERTa** remains the primary fallback because it is smaller, faster, and already closely aligned with the September pipeline.

**RoBERTa-base** is retained as a reference candidate, but it currently offers neither the compute advantage of DistilRoBERTa nor the legal-domain specialization of Legal-BERT.

The encoder is **not frozen yet**. The final selection will be made after the token-length and truncation audit in Step 3.

CHECK NOTES AT THE BOTTOM TO UNDERSTAND THE TOKENS AND ENCODERS CONSIDERED.


## Step 3: Token-Length and Truncation Audit

Different tokenizers can split the same contract text into different numbers of tokens.

This step compares the approved paragraph chunks using:

- DistilRoBERTa
- RoBERTa-base
- Legal-BERT

For each tokenizer, we measure average token length, maximum length, and how many chunks exceed the 512-token limit.

### Truncation Policy

- Chunks within the limit are kept unchanged.
- Oversized chunks are re-split using the selected tokenizer.
- `truncation=True` is used only as a final safety check.

The goal is to choose a tokenizer that fits the approved chunks while minimizing lost contract text.

In [63]:
# Inspect the approved paragraph chunks

print("Chunk shape:", train_chunks.shape)

print("\nAvailable columns:")
print(train_chunks.columns.tolist())

display(train_chunks.head())

Chunk shape: (10841, 11)

Available columns:
['method', 'contract_id', 'context_group_id', 'chunk_id', 'chunk_number', 'char_start', 'char_end', 'num_tokens', 'chunk_text', 'chunk_hash', 'document_hash']


,method,contract_id,context_group_id,chunk_id,chunk_number,char_start,char_end,num_tokens,chunk_text,chunk_hash,document_hash
0,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0000,0,0,2056,463,EXHIBIT 10.6\n\n ...,8d9734236056eaa8a3619bf2942ca9b27adadb920c52ff...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...
1,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0001,1,2076,3126,512,1.1 Grant and Acceptance. Company h...,0eb8801fa97ea183343a724bb6fbd0caab779ccfcdb942...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...
2,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0002,2,2979,3717,359,such customers in the Market to those custom...,4e070f16a388ecff6151b6344f9ff838669ee88d9e2424...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...
3,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0003,3,3737,4788,512,1.2 License. The Company hereby grants ...,fe8b1735b9cc0a5536b6771e0f47e30c7c80c93992179d...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...
4,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0004,4,4658,5228,277,sublicense the Names or to do ...,c2b10a8c277aef96161b58f42a2068a32e017068e12f31...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...


In [64]:
# Identify the column containing the chunk text

possible_text_columns = [
    "text",
    "chunk_text",
    "context",
    "content"
]

TEXT_COLUMN = next(
    (
        column
        for column in possible_text_columns
        if column in train_chunks.columns
    ),
    None
)

if TEXT_COLUMN is None:
    raise ValueError(
        "Could not automatically identify the chunk-text column. "
        f"Available columns: {train_chunks.columns.tolist()}"
    )

print("Chunk text column:", TEXT_COLUMN)

# Make sure no chunk text is missing.
assert train_chunks[TEXT_COLUMN].notna().all()

print("PASS: Chunk text is available for tokenizer audit.")

Chunk text column: chunk_text
PASS: Chunk text is available for tokenizer audit.


In [65]:
# Token-length audit function

def audit_tokenizer_lengths(
    chunks_df,
    tokenizer,
    encoder_name,
    text_column
):
    """
    Measure the token length of every approved paragraph chunk
    without truncating the text.

    Returns one row per chunk so that oversized chunks can be
    inspected later.
    """

    records = []

    # Determine the input limit supported by this tokenizer/model.
    model_limit = tokenizer.model_max_length

    for row_number, (_, row) in enumerate(chunks_df.iterrows()):

        text = str(row[text_column])

        # Tokenize the COMPLETE chunk.
        #
        # truncation=False is essential here because this is an audit:
        # we need to know the true length before anything is removed.
        encoded = tokenizer(
            text,
            add_special_tokens=True,
            truncation=False,
            verbose=False
        )

        token_length = len(encoded["input_ids"])

        # Measure how far the chunk exceeds the model limit.
        excess_tokens = max(
            token_length - model_limit,
            0
        )

        record = {
            "encoder": encoder_name,
            "row_number": row_number,
            "token_length": token_length,
            "model_limit": model_limit,
            "exceeds_limit": token_length > model_limit,
            "excess_tokens": excess_tokens
        }

        # Preserve useful identifiers when available.
        if "chunk_id" in chunks_df.columns:
            record["chunk_id"] = row["chunk_id"]

        if "contract_id" in chunks_df.columns:
            record["contract_id"] = row["contract_id"]

        records.append(record)

    return pd.DataFrame(records)

In [66]:
# Run the audit for all candidate tokenizers

token_length_results = []

for encoder_name, tokenizer in candidate_tokenizers.items():

    print(f"Auditing {encoder_name}...")

    result = audit_tokenizer_lengths(
        chunks_df=train_chunks,
        tokenizer=tokenizer,
        encoder_name=encoder_name,
        text_column=TEXT_COLUMN
    )

    token_length_results.append(result)


# Combine all tokenizer results.
token_length_audit = pd.concat(
    token_length_results,
    ignore_index=True
)

print("\nPASS: Token-length audit completed.")

print("Total audit rows:", len(token_length_audit))

Auditing DistilRoBERTa...
Auditing RoBERTa-base...
Auditing Legal-BERT...

PASS: Token-length audit completed.
Total audit rows: 32523


In [67]:
# Summarize token lengths and truncation risk

audit_summary = (
    token_length_audit
    .groupby("encoder")
    .agg(
        total_chunks=("token_length", "count"),
        mean_tokens=("token_length", "mean"),
        median_tokens=("token_length", "median"),
        p95_tokens=(
            "token_length",
            lambda x: x.quantile(0.95)
        ),
        max_tokens=("token_length", "max"),
        chunks_over_limit=("exceeds_limit", "sum"),
        total_excess_tokens=("excess_tokens", "sum")
    )
    .reset_index()
)


# Calculate the percentage of chunks that exceed
# the model's maximum input length.
audit_summary["pct_over_limit"] = (
    audit_summary["chunks_over_limit"]
    / audit_summary["total_chunks"]
    * 100
)


# Round values for readability.
audit_summary[
    [
        "mean_tokens",
        "median_tokens",
        "p95_tokens",
        "pct_over_limit"
    ]
] = audit_summary[
    [
        "mean_tokens",
        "median_tokens",
        "p95_tokens",
        "pct_over_limit"
    ]
].round(2)


# Arrange columns clearly.
audit_summary = audit_summary[
    [
        "encoder",
        "total_chunks",
        "mean_tokens",
        "median_tokens",
        "p95_tokens",
        "max_tokens",
        "chunks_over_limit",
        "pct_over_limit",
        "total_excess_tokens"
    ]
]


display(audit_summary)

,encoder,total_chunks,mean_tokens,median_tokens,p95_tokens,max_tokens,chunks_over_limit,pct_over_limit,total_excess_tokens
0,DistilRoBERTa,10841,398.32,468.0,557.0,1035,1896,17.49,78751
1,Legal-BERT,10841,335.28,376.0,517.0,1534,688,6.35,42764
2,RoBERTa-base,10841,398.32,468.0,557.0,1035,1896,17.49,78751


In [68]:
# Inspect chunks that exceed the model limit

overflow_chunks = token_length_audit[
    token_length_audit["exceeds_limit"]
].copy()


if len(overflow_chunks) == 0:

    print("PASS: No chunks exceed the model input limit.")

else:

    print(
        "Total tokenizer/chunk combinations over limit:",
        len(overflow_chunks)
    )

    overflow_summary = (
        overflow_chunks
        .groupby("encoder")
        .agg(
            oversized_chunks=("token_length", "count"),
            smallest_overflow=("token_length", "min"),
            largest_overflow=("token_length", "max"),
            avg_excess_tokens=("excess_tokens", "mean")
        )
        .round(2)
        .reset_index()
    )

    display(overflow_summary)

Total tokenizer/chunk combinations over limit: 4480


,encoder,oversized_chunks,smallest_overflow,largest_overflow,avg_excess_tokens
0,DistilRoBERTa,1896,513,1035,41.54
1,Legal-BERT,688,513,1534,62.16
2,RoBERTa-base,1896,513,1035,41.54


In [69]:
# Estimate information loss from naive truncation

truncation_impact = (
    token_length_audit
    .groupby("encoder")
    .agg(
        total_tokens=("token_length", "sum"),
        tokens_that_would_be_removed=("excess_tokens", "sum")
    )
    .reset_index()
)


truncation_impact["pct_tokens_lost"] = (
    truncation_impact["tokens_that_would_be_removed"]
    / truncation_impact["total_tokens"]
    * 100
).round(4)


display(truncation_impact)

,encoder,total_tokens,tokens_that_would_be_removed,pct_tokens_lost
0,DistilRoBERTa,4318213,78751,1.8237
1,Legal-BERT,3634761,42764,1.1765
2,RoBERTa-base,4318213,78751,1.8237


In [70]:
# Check stored token counts against fresh DistilRoBERTa counts

distil_audit = token_length_audit[
    token_length_audit["encoder"] == "DistilRoBERTa"
].copy()

# Match the fresh audit back to the original chunk metadata
token_count_check = train_chunks[
    ["chunk_id", "num_tokens", "chunk_text"]
].merge(
    distil_audit[
        ["chunk_id", "token_length"]
    ],
    on="chunk_id",
    how="left"
)

# Difference between the token count recorded during chunking
# and the token count obtained by retokenizing the final chunk text.
token_count_check["token_difference"] = (
    token_count_check["token_length"]
    - token_count_check["num_tokens"]
)

print(
    "Chunks with different stored vs. retokenized counts:",
    (token_count_check["token_difference"] != 0).sum()
)

print(
    "Largest token-count difference:",
    token_count_check["token_difference"].max()
)

print(
    "Average token-count difference:",
    round(token_count_check["token_difference"].mean(), 2)
)


# Show the largest mismatches
display(
    token_count_check
    .sort_values(
        "token_difference",
        ascending=False
    )
    .head(10)
)

Chunks with different stored vs. retokenized counts: 6964
Largest token-count difference: 602
Average token-count difference: 17.78


,chunk_id,num_tokens,chunk_text,token_length,token_difference
7540,contract_0280__chunk_0086,217,Without limiting Certegy's requirements under ...,819,602
2101,contract_0086__chunk_0030,496,[*]=CERTAIN INFORMATION IN THIS EXHIBIT HAS BE...,1035,539
2098,contract_0086__chunk_0027,457,[*]=CERTAIN INFORMATION IN THIS EXHIBIT HAS BE...,949,492
2099,contract_0086__chunk_0028,457,[*]=CERTAIN INFORMATION IN THIS EXHIBIT HAS BE...,946,489
6091,contract_0228__chunk_0093,510,[…***…]\n\n\n\n\n\n\n\n\n\n […***…]\n\n\n\n\n\...,995,485
6092,contract_0228__chunk_0094,504,[…***…]\n\n\n\n\n\n […***…]\n\n\n\n\n\n\n\n\n\...,987,483
2100,contract_0086__chunk_0029,457,[*]=CERTAIN INFORMATION IN THIS EXHIBIT HAS BE...,940,483
7509,contract_0280__chunk_0055,511,"License Agreement between Equifax, Inc. Equi...",988,477
1684,contract_0068__chunk_0058,476,For and on behalf of Metromedia Fiber Network ...,930,454
6000,contract_0228__chunk_0002,508,4.2\n\n Cancellation of Order\n\n 17\n\n\n\n...,945,437


In [71]:
# How many chunks were recorded as model-safe
# but exceed 512 when the actual text is retokenized?
unexpected_overflow = token_count_check[
    (token_count_check["num_tokens"] <= 512)
    &
    (token_count_check["token_length"] > 512)
]

print(
    "Chunks stored as <=512 but actually >512:",
    len(unexpected_overflow)
)

display(
    unexpected_overflow[
        [
            "chunk_id",
            "num_tokens",
            "token_length",
            "token_difference"
        ]
    ].head(10)
)

Chunks stored as <=512 but actually >512: 1896


,chunk_id,num_tokens,token_length,token_difference
0,contract_0001__chunk_0000,463,582,119
7,contract_0001__chunk_0007,495,578,83
8,contract_0001__chunk_0008,442,550,108
34,contract_0001__chunk_0034,481,519,38
41,contract_0001__chunk_0041,505,552,47
43,contract_0001__chunk_0043,487,514,27
52,contract_0001__chunk_0052,486,549,63
53,contract_0001__chunk_0053,473,547,74
59,contract_0002__chunk_0001,510,514,4
60,contract_0002__chunk_0002,497,524,27


### Token-Count Consistency Check

The audit identified a mismatch between the token counts stored during paragraph chunking and the token counts obtained by retokenizing the final `chunk_text`.

For DistilRoBERTa:

- **6,964 of 10,841 chunks** had a different stored and retokenized token count.
- The average difference was **17.78 tokens**.
- The largest difference was **602 tokens**.
- **1,896 chunks** were stored as being within the 512-token limit but exceeded 512 when the actual final text was retokenized.

This means the historical `num_tokens` field should not be used to determine whether a chunk is model-safe. For downstream modeling, token length will be calculated directly from the final `chunk_text` using the tokenizer paired with the selected encoder.

The discrepancy likely results from paragraph-level token counts being combined while the final chunk text contains additional formatting and text between paragraph boundaries.

Therefore, the October pipeline will consider **fresh tokenizer-specific token counts** as the source of truth.

In [72]:
# Freeze token-length source of truth

TOKEN_LENGTH_SOURCE = "fresh_tokenization_of_chunk_text"
MAX_MODEL_LENGTH = 512

print("Token-length source:", TOKEN_LENGTH_SOURCE)
print("Maximum model input length:", MAX_MODEL_LENGTH)
print(
    "Policy: Oversized chunks will be re-split before training, "
    "not silently truncated."
)

Token-length source: fresh_tokenization_of_chunk_text
Maximum model input length: 512
Policy: Oversized chunks will be re-split before training, not silently truncated.


### Step 3 Conclusion

**Legal-BERT is the best fit** for the approved paragraph chunks.

- Average length: **335.28 tokens**
- Chunks over 512 tokens: **688 of 10,841 (6.35%)**
- Estimated token loss from truncation: **1.18%**
- DistilRoBERTa and RoBERTa-base each had **1,896 oversized chunks (17.49%)**

The audit also showed that the stored `num_tokens` values are not always accurate. Therefore, the October pipeline will use **fresh tokenization of the final `chunk_text`** as the source of truth.

### Truncation Strategy

- Retokenize each chunk with the selected tokenizer.
- Re-split chunks over 512 tokens.
- Use `truncation=True` only as a final safety check.

Based on legal-domain relevance and token efficiency, **Legal-BERT remains the leading encoder/tokenizer candidate**.

In [73]:
# Step 3 locked decisions

LEADING_ENCODER = "Legal-BERT"
LEADING_MODEL_ID = "nlpaueb/legal-bert-base-uncased"

TOKEN_LENGTH_SOURCE = "fresh_tokenization_of_chunk_text"
MAX_MODEL_LENGTH = 512
TRUNCATION_POLICY = "re-split oversized chunks before training"

print("Leading encoder:", LEADING_ENCODER)
print("Model ID:", LEADING_MODEL_ID)
print("Token-length source:", TOKEN_LENGTH_SOURCE)
print("Maximum input length:", MAX_MODEL_LENGTH)
print("Truncation policy:", TRUNCATION_POLICY)

Leading encoder: Legal-BERT
Model ID: nlpaueb/legal-bert-base-uncased
Token-length source: fresh_tokenization_of_chunk_text
Maximum input length: 512
Truncation policy: re-split oversized chunks before training


## Step 4: Pre-register Training Configuration

Before training, we fix the main settings to keep experiments consistent and reproducible.

### Training Plan

- **Primary encoder:** Legal-BERT
- **Fine-tuning:** LoRA to reduce Colab memory use
- **Seeds:** `42`, `123`, `2026`
- **Run cap:** 3 completed runs, one per seed
- **Imbalance treatment:** Positive-class weighting with binary cross-entropy loss
- **Class weights:** Calculated from training data only

For each category:

`positive weight = negatives / positives`

### Compute Fallback

If Legal-BERT + LoRA exceeds Colab limits:

1. Reduce batch size
2. Increase gradient accumulation
3. Use mixed precision if supported
4. Fall back to DistilRoBERTa if needed

The fallback is based on compute limitations, not model performance.

CHECK NOTES AT THE BOTTOM TO UNDERSTAND WHY THESE DECISIONS WERE MADE


In [74]:
# Step 4: Pre-register training configuration

TRAINING_CONFIG = {

    # Encoder
    "primary_encoder": "Legal-BERT",
    "primary_model_id": "nlpaueb/legal-bert-base-uncased",

    "fallback_encoder": "DistilRoBERTa",
    "fallback_model_id": "distilbert/distilroberta-base",

    # Multi-label classification across the 10 CUAD categories
    "num_labels": len(FINAL_IDS),
    "problem_type": "multi_label_classification",

    # Reproducibility
    "seeds": [42, 123, 2026],
    "max_completed_runs": 3,

    # Input handling
    "max_length": 512,
    "token_length_source": "fresh_tokenization_of_chunk_text",
    "oversized_chunk_policy": "re-split_before_training",

    # Parameter-efficient fine-tuning
    "fine_tuning_method": "LoRA",

    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.10,

    # BERT-family attention projection layers
    "lora_target_modules": [
        "query",
        "value"
    ],

    # Imbalance treatment
    "imbalance_treatment": "positive_class_weighting",
    "loss_function": "BCEWithLogitsLoss",

    # Compute fallback
    "fallback_order": [
        "reduce_per_device_batch_size",
        "increase_gradient_accumulation",
        "enable_mixed_precision_if_supported",
        "fallback_to_DistilRoBERTa"
    ]
}


# Display the registered settings clearly.
training_config_df = pd.DataFrame(
    TRAINING_CONFIG.items(),
    columns=["setting", "registered_value"]
)

display(training_config_df)

,setting,registered_value
0,primary_encoder,Legal-BERT
1,primary_model_id,nlpaueb/legal-bert-base-uncased
2,fallback_encoder,DistilRoBERTa
3,fallback_model_id,distilbert/distilroberta-base
4,num_labels,10
5,problem_type,multi_label_classification
6,seeds,"[42, 123, 2026]"
7,max_completed_runs,3
8,max_length,512
9,token_length_source,fresh_tokenization_of_chunk_text


### LoRA Configuration

The initial LoRA configuration is intentionally conservative:

- **Rank (`r`) = 8:** adds limited trainable capacity while keeping memory use low.
- **Alpha = 16:** scales the LoRA updates relative to the selected rank.
- **Dropout = 0.10:** provides regularization during fine-tuning.
- **Target modules = `query` and `value`:** applies LoRA to the attention projections commonly adapted in BERT-family models.

These settings are pre-registered and will not be changed during the three primary runs unless a technical incompatibility prevents training.

In [75]:
# Reproducibility helper

import random
import numpy as np
import torch


def set_training_seed(seed):
    """
    Set random seeds across Python, NumPy, and PyTorch
    for more reproducible model training.
    """

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


# Verify the registered seeds.
REGISTERED_SEEDS = TRAINING_CONFIG["seeds"]

print("Registered training seeds:", REGISTERED_SEEDS)
print("Maximum completed runs:", TRAINING_CONFIG["max_completed_runs"])

Registered training seeds: [42, 123, 2026]
Maximum completed runs: 3


In [76]:
# Registered class-imbalance treatment

def calculate_positive_class_weights(label_matrix):
    """
    Calculate one positive-class weight for each CUAD category.

    Formula:
        pos_weight = number of negatives / number of positives

    Parameters
    ----------
    label_matrix:
        Binary training-label matrix with shape:
        (number of training examples, number of categories)

    Returns
    -------
    torch.Tensor
        One positive weight per category.

    Important:
        These weights must be calculated using TRAINING labels only.
        Validation and test labels must not influence the weights.
    """

    labels = np.asarray(label_matrix)

    positive_counts = labels.sum(axis=0)
    negative_counts = len(labels) - positive_counts

    # Prevent division by zero if a category unexpectedly
    # contains no positive training examples.
    if np.any(positive_counts == 0):
        raise ValueError(
            "At least one registered category has zero "
            "positive examples in the training data."
        )

    positive_weights = (
        negative_counts / positive_counts
    )

    return torch.tensor(
        positive_weights,
        dtype=torch.float32
    )

### Step 4 Conclusion

The training protocol is now pre-registered.

- **Primary encoder:** Legal-BERT
- **Fine-tuning method:** LoRA
- **LoRA configuration:** `r=8`, `alpha=16`, `dropout=0.10`, targeting attention `query` and `value`
- **Seeds:** 42, 123, and 2026
- **Run cap:** 3 completed primary runs
- **Imbalance treatment:** training-only positive-class weighting with `BCEWithLogitsLoss`
- **Maximum input length:** 512 tokens
- **Oversized chunks:** re-split before training rather than silently truncated
- **Fallback encoder:** DistilRoBERTa, used only if Legal-BERT cannot run reliably within the available compute budget

These settings will remain fixed during the registered runs so that later model comparisons reflect model performance rather than repeated configuration changes.

## Step 5: Define Model Selection and Threshold Rules

Before training, the selection rules are fixed to avoid choosing results after seeing which model performs best.

### Model Selection

The primary metric is **validation Macro PR-AUC** because the task is multi-label and highly imbalanced. Macro averaging gives equal importance to all 10 categories.

The model with the highest validation Macro PR-AUC will be selected.

### Threshold Selection

Each category will have its own threshold:

- Tuned on **validation data only**
- Searched from `0.05` to `0.95` in steps of `0.05`
- Selected based on the highest category F1 score
- Frozen before final evaluation

The test set will not be used for model or threshold selection.

### Final Metrics

The selected model will report:

- Macro Precision, Recall, F1
- Micro Precision, Recall, F1
- Macro PR-AUC

In [77]:
# Step 5: Pre-register model-selection and threshold rules

import numpy as np

MODEL_SELECTION_CONFIG = {

    # Primary model-selection rule
    "primary_selection_metric": "macro_pr_auc",
    "selection_split": "validation",
    "selection_direction": "maximize",

    # Threshold tuning
    "threshold_strategy": "per_category",
    "threshold_metric": "f1",
    "threshold_split": "validation",

    # Fixed threshold search grid
    "threshold_grid": np.round(
        np.arange(0.05, 1.00, 0.05),
        2
    ).tolist(),

    # Evaluation metrics reported after thresholds are frozen
    "reported_metrics": [
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "micro_precision",
        "micro_recall",
        "micro_f1",
        "macro_pr_auc"
    ],

    # Test-data policy
    "official_test_for_selection": False,
    "official_test_for_threshold_tuning": False
}

for setting, value in MODEL_SELECTION_CONFIG.items():
    print(f"{setting}: {value}")

primary_selection_metric: macro_pr_auc
selection_split: validation
selection_direction: maximize
threshold_strategy: per_category
threshold_metric: f1
threshold_split: validation
threshold_grid: [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
reported_metrics: ['macro_precision', 'macro_recall', 'macro_f1', 'micro_precision', 'micro_recall', 'micro_f1', 'macro_pr_auc']
official_test_for_selection: False
official_test_for_threshold_tuning: False


In [78]:
# Per-category threshold-selection rule

from sklearn.metrics import f1_score


def select_category_thresholds(
    y_true,
    y_prob,
    categories,
    threshold_grid=None
):
    """
    Select one threshold per category using validation data only.

    For each category:
        1. Test every registered threshold.
        2. Calculate F1.
        3. Keep the threshold with the highest F1.

    Parameters
    ----------
    y_true:
        Binary validation labels with shape
        (n_examples, n_categories).

    y_prob:
        Predicted probabilities with the same shape.

    categories:
        Ordered list of category names.

    threshold_grid:
        Pre-registered threshold values.

    Returns
    -------
    dict
        Selected threshold for each category.
    """

    if threshold_grid is None:
        threshold_grid = MODEL_SELECTION_CONFIG["threshold_grid"]

    selected_thresholds = {}

    for category_index, category in enumerate(categories):

        true_labels = y_true[:, category_index]
        probabilities = y_prob[:, category_index]

        best_threshold = None
        best_f1 = -1

        for threshold in threshold_grid:

            predictions = (
                probabilities >= threshold
            ).astype(int)

            score = f1_score(
                true_labels,
                predictions,
                zero_division=0
            )

            if score > best_f1:
                best_f1 = score
                best_threshold = threshold

        selected_thresholds[category] = {
            "threshold": best_threshold,
            "validation_f1": round(best_f1, 4)
        }

    return selected_thresholds

In [79]:
# September logistic-regression baseline

SEPTEMBER_BASELINE = {

    "model": "DistilRoBERTa embeddings + Logistic Regression",

    # Task 12 validation results
    "validation_macro_precision": 0.505,
    "validation_macro_recall": 0.582,
    "validation_macro_f1": 0.535,
    "validation_macro_pr_auc": 0.529,

    # Task 13 official-test results
    # Reference only — not used for October model selection.
    "test_macro_precision": 0.4636,
    "test_macro_recall": 0.5813,
    "test_macro_f1": 0.5112
}

baseline_df = pd.DataFrame(
    SEPTEMBER_BASELINE.items(),
    columns=["metric", "baseline_value"]
)

display(baseline_df)

,metric,baseline_value
0,model,DistilRoBERTa embeddings + Logistic Regression
1,validation_macro_precision,0.505
2,validation_macro_recall,0.582
3,validation_macro_f1,0.535
4,validation_macro_pr_auc,0.529
5,test_macro_precision,0.4636
6,test_macro_recall,0.5813
7,test_macro_f1,0.5112


### Multi-Seed Reporting Rule

The Legal-BERT + LoRA configuration will be trained using the three registered seeds: `42`, `123`, and `2026`.

The configuration will be evaluated across all three runs. The primary reported result will be the **mean validation Macro PR-AUC across the three seeds**, along with its standard deviation.

Individual seed results will also be retained for reproducibility.

Thresholds will be derived from validation predictions only and frozen before any official-test evaluation.

### Step 5 Conclusion

The evaluation rules are now pre-registered:

- **Primary selection metric:** Validation Macro PR-AUC
- **Threshold method:** One validation-tuned threshold per CUAD category
- **Threshold objective:** Maximum category-level F1
- **Threshold search:** 0.05–0.95 in 0.05 increments
- **Main post-threshold metric:** Macro F1
- **Additional metrics:** Macro/Micro Precision, Recall, F1, and Macro PR-AUC
- **Multi-seed reporting:** Mean and standard deviation across seeds 42, 123, and 2026
- **Baseline comparison:** September logistic-regression baseline
- **Official test set:** Never used for model selection or threshold tuning

The existing validation baseline to compare against is **0.529 Macro PR-AUC and 0.535 Macro F1**. The official-test Macro F1 of **0.5112** is retained only as a later generalization reference.

These rules will remain fixed before model training and evaluation begin.

## Step 6: Document and Freeze the Setup

The final step is to save the decisions made in this task so that model training starts from a fixed specification.

The frozen setup records:

- Selected encoder and tokenizer
- Fallback encoder
- Maximum token length and truncation policy
- Random seeds and run cap
- LoRA configuration
- Class-imbalance treatment
- Model-selection metric
- Threshold-selection rules
- Token-length audit results
- Rationale for the main decisions

In [80]:
# STEP 6: DOCUMENT AND FREEZE THE TASK 1 SETUP

import os
import json
import pandas as pd

# 1. Create an output folder for the frozen Task 1 artifacts

OUTPUT_DIR = "october_task1_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# 2. Define the frozen model specification

frozen_spec = {

    # Project scope
    "task": "October Task 1 - Encoder, Tokenizer, and Model Pre-registration",

    "categories": [
        "governing_law",
        "renewal_term",
        "revenue_profit_sharing",
        "cap_on_liability",
        "uncapped_liability",
        "termination_for_convenience",
        "anti_assignment",
        "audit_rights",
        "license_grant",
        "exclusivity"
    ],

    # Encoder / tokenizer
    "encoder": {
        "primary_name": "Legal-BERT",
        "primary_model_id": "nlpaueb/legal-bert-base-uncased",

        "fallback_name": "DistilRoBERTa",
        "fallback_model_id": "distilbert/distilroberta-base",

        "selection_rationale":
            "Legal-BERT was selected because it is pretrained on legal text "
            "and produced fewer chunks over the 512-token limit during the "
            "token-length audit."
    },

    # Tokenization
    "tokenization": {
        "max_length": 512,

        "token_length_source":
            "Fresh tokenization of final chunk_text using the selected tokenizer",

        "oversized_chunk_policy":
            "Re-split chunks over 512 tokens before training",

        "truncation":
            "Use truncation=True only as a final safety guard",

        "rationale":
            "Stored historical token counts did not always match token counts "
            "from the final chunk text, so fresh tokenizer-specific counts are "
            "used as the source of truth."
    },

    # Training configuration
    "training": {
        "random_seeds": [42, 123, 2026],

        "maximum_completed_runs": 3,

        "fine_tuning_method": "LoRA",

        "lora": {
            "r": 8,
            "alpha": 16,
            "dropout": 0.10,
            "target_modules": ["query", "value"]
        },

        "loss_function": "BCEWithLogitsLoss",

        "imbalance_treatment":
            "Positive-class weighting",

        "positive_weight_formula":
            "number of negative training examples / number of positive training examples",

        "weight_data_source":
            "Training data only"
    },

    # Compute fallback
    "compute_fallback": {
        "trigger":
            "Compute or memory limitations only, not model performance",

        "steps": [
            "Reduce per-device batch size",
            "Increase gradient accumulation",
            "Enable mixed-precision training when supported",
            "Fall back to DistilRoBERTa if Legal-BERT still cannot train reliably"
        ]
    },

    # Model selection
    "model_selection": {
        "primary_metric": "Validation Macro PR-AUC",

        "seed_reporting":
            "Report individual seed results plus mean and standard deviation",

        "test_set_usage":
            "Official test data cannot be used for model selection"
    },

    # Threshold rules
    "threshold_selection": {
        "method": "Per-category validation threshold",

        "selection_metric": "F1",

        "grid_start": 0.05,
        "grid_end": 0.95,
        "grid_step": 0.05,

        "data_source": "Validation predictions only",

        "final_thresholds":
            "To be selected after training using the frozen validation rule",

        "test_set_usage":
            "Official test data cannot be used to tune thresholds"
    },

    # Final evaluation metrics
    "evaluation_metrics": [
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Micro Precision",
        "Micro Recall",
        "Micro F1",
        "Macro PR-AUC"
    ]
}


# 3. Add the tokenizer-audit results

frozen_spec["tokenizer_audit"] = {

    "DistilRoBERTa": {
        "mean_tokens": 398.32,
        "median_tokens": 468,
        "p95_tokens": 557,
        "max_tokens": 1035,
        "chunks_over_512": 1896,
        "percent_over_512": 17.49,
        "total_excess_tokens": 78751
    },

    "RoBERTa-base": {
        "mean_tokens": 398.32,
        "median_tokens": 468,
        "p95_tokens": 557,
        "max_tokens": 1035,
        "chunks_over_512": 1896,
        "percent_over_512": 17.49,
        "total_excess_tokens": 78751
    },

    "Legal-BERT": {
        "mean_tokens": 335.28,
        "median_tokens": 376,
        "p95_tokens": 517,
        "max_tokens": 1534,
        "chunks_over_512": 688,
        "percent_over_512": 6.35,
        "total_excess_tokens": 42764
    },

    "token_count_consistency_check": {
        "chunks_checked": 10841,
        "distilroberta_mismatched_counts": 6964,
        "average_difference_tokens": 17.78,
        "largest_difference_tokens": 602,
        "stored_under_512_but_retokenized_over_512": 1896
    }
}


# 4. Save the frozen specification as JSON

spec_path = os.path.join(
    OUTPUT_DIR,
    "task1_frozen_model_specification.json"
)

with open(spec_path, "w", encoding="utf-8") as f:
    json.dump(frozen_spec, f, indent=4)


print("Frozen specification saved to:")
print(spec_path)

from google.colab import files
with open("task1_frozen_model_specification.json", "w") as f:
    json.dump(frozen_spec, f, indent=4)

files.download("task1_frozen_model_specification.json")

Frozen specification saved to:
october_task1_outputs/task1_frozen_model_specification.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [81]:
from google.colab import files
# Save the tokenizer audit separately for easy review

audit_path = os.path.join(
    OUTPUT_DIR,
    "task1_tokenizer_audit_summary.csv"
)

audit_summary.to_csv(audit_path, index=False)

print("Tokenizer audit saved to:")
print(audit_path)

audit_summary.to_csv(
    "task1_tokenizer_audit_summary.csv",
    index=False
)
files.download("task1_tokenizer_audit_summary.csv")

Tokenizer audit saved to:
october_task1_outputs/task1_tokenizer_audit_summary.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [82]:
# VERIFY THE FROZEN SETUP

with open(
    "october_task1_outputs/task1_frozen_model_specification.json",
    "r",
    encoding="utf-8"
) as f:
    frozen_check = json.load(f)

print("Primary encoder:",
      frozen_check["encoder"]["primary_name"])

print("Model ID:",
      frozen_check["encoder"]["primary_model_id"])

print("Seeds:",
      frozen_check["training"]["random_seeds"])

print("Run cap:",
      frozen_check["training"]["maximum_completed_runs"])

print("LoRA configuration:",
      frozen_check["training"]["lora"])

print("Selection metric:",
      frozen_check["model_selection"]["primary_metric"])

print("Threshold rule:",
      frozen_check["threshold_selection"]["method"])

print("\n✅ Task 1 configuration successfully frozen.")

Primary encoder: Legal-BERT
Model ID: nlpaueb/legal-bert-base-uncased
Seeds: [42, 123, 2026]
Run cap: 3
LoRA configuration: {'r': 8, 'alpha': 16, 'dropout': 0.1, 'target_modules': ['query', 'value']}
Selection metric: Validation Macro PR-AUC
Threshold rule: Per-category validation threshold

✅ Task 1 configuration successfully frozen.


**___________________________________________**

## Notes to Understand Encoders and Tokenizers

Before training the next model, we compare three realistic transformer options: **DistilRoBERTa, RoBERTa-base, and Legal-BERT**.

A **tokenizer** is the part of the NLP pipeline that breaks text into smaller pieces called **tokens** so that a model can process the text. A token may be a full word, part of a word, punctuation, or another text unit.

An **encoder** is the pretrained language model that reads those tokens and turns them into numerical representations that capture their meaning and context. These representations can then be used for classification.

In simple terms:

- **DistilRoBERTa** is a smaller and faster version of RoBERTa. It is trained on general English and is useful when compute resources are limited.
- **RoBERTa-base** is a larger general-purpose language model. It has more layers and parameters than DistilRoBERTa, which gives it more modeling capacity but also makes it more expensive to train.
- **Legal-BERT** is a BERT-based model that was pretrained specifically on legal text. Because our project focuses on commercial contracts, this makes it especially relevant for the task.

The first code block simply **registers the three candidates and their basic characteristics** so they can be compared consistently.

The fields mean:

- `model_id`: the Hugging Face identifier used to load the model/tokenizer
- `domain`: the type of language used during pretraining
- `parameters_millions`: approximately how many trainable parameters the model contains
- `layers`: the number of transformer layers in the encoder
- `expected_max_length`: the expected maximum input length in tokens

### How the Qualitative Comparison Was Decided

The second table is a **reasoned comparison**, not a result from model training.

The ratings are based on three practical criteria for this project:

1. **Legal-text fit**  
   This asks how closely the model's pretraining matches our contract data.
   - DistilRoBERTa and RoBERTa-base receive **Moderate** because they are general-language models.
   - Legal-BERT receives **Strong** because it was pretrained on legal text, making it more aligned with contract language.

2. **Compute fit**  
   This considers how realistic the model is to train in our Google Colab environment.
   - DistilRoBERTa receives **Strong** because it is the smallest candidate, with about 82M parameters and 6 layers.
   - RoBERTa-base and Legal-BERT receive **Moderate** because they are larger, with about 110–125M parameters and 12 layers, so they require more memory and computation.

3. **Ease of training**  
   All three receive **Strong** because they are widely supported through Hugging Face and can be used with standard transformer-training tools, including parameter-efficient methods such as LoRA.

The `strength` and `tradeoff` columns summarize the main reason each model may or may not be a good fit.

These ratings are therefore based on **model architecture, pretraining domain, expected compute requirements, and compatibility with our project setup**. They are preliminary judgments only. The final encoder choice should also consider the token-length audit and downstream model performance.

## Why These Training Decisions Were Chosen

#### Seeds: `42`, `123`, `2026`
Using multiple random seeds helps us check whether the model performs consistently or whether the result depends too much on randomness.

- `42` is a common reproducibility seed.
- `123` gives us a second independent run.
- `2026` matches the project year and gives us a third independent run.

Using three seeds gives us a better sense of stability without creating too many expensive training runs.

#### Run Cap: 3 Runs
We cap training at **3 completed runs**, one for each seed.

This keeps the experiment manageable in Google Colab and prevents us from repeatedly retraining until we get a result we like. It also makes the experiment easier to reproduce because everyone knows exactly how many runs were allowed.

#### Primary Encoder: Legal-BERT
Legal-BERT is chosen as the primary encoder because it is pretrained on legal text, which makes it a better match for commercial contracts than a general-language model.

It also performed better in our token-length audit, with fewer chunks exceeding the 512-token limit.

#### Fine-Tuning Method: LoRA
LoRA is chosen because full fine-tuning of a transformer can require a lot of GPU memory.

LoRA trains only a small number of added parameters while keeping most of the pretrained model frozen. This makes training more practical in Google Colab while still allowing the model to adapt to our contract-classification task.

#### LoRA Settings
The initial settings are:

- `r = 8`
- `alpha = 16`
- `dropout = 0.10`
- target modules: `query` and `value`

These are conservative settings designed to keep the number of trainable parameters small while still giving the model enough flexibility to learn the task.

They are being pre-registered so they are not changed repeatedly after seeing results.

#### Class-Imbalance Treatment: Positive-Class Weighting
Our dataset has many more negative examples than positive examples for several categories.

Positive-class weighting gives more importance to mistakes on rare positive examples.

The weight is calculated as:

`positive weight = number of negatives / number of positives`

This helps the model pay more attention to underrepresented positive classes without changing the dataset through oversampling or undersampling.

Weights are calculated using **training data only** so that validation information does not leak into training.

#### Loss Function: Binary Cross-Entropy with Logits
The task is **multi-label classification**, meaning one chunk can belong to more than one category.

`BCEWithLogitsLoss` is appropriate because it treats each category as a separate yes/no prediction and works directly with positive-class weights.

#### Fallback Plan
The fallback plan is designed around compute limits, not model performance.

If Legal-BERT + LoRA does not fit reliably in Colab:

1. Reduce batch size
2. Increase gradient accumulation
3. Use mixed precision if supported
4. Fall back to DistilRoBERTa

This keeps the training plan practical while avoiding changing models simply because one performs better after seeing results.